# Кросс-валидация в `pytorch`

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import matplotlib.pyplot as plt

import pandas_datareader.data as web

In [ ]:
# Подготовка датасета для обучения в формате inputs, outputs
def make_datasets(input_data, n_inputs=2, n_outputs=1, gap=0):
	L = len(input_data)
	y = np.full((L-n_inputs-n_outputs-gap, n_outputs), 0.0)
	X = np.full((L-n_inputs-n_outputs-gap, n_inputs), 0.0)

	for i in range(n_inputs):
		X[:,i] = input_data[i:L-n_inputs-n_outputs-gap+i]

	for i in range(n_outputs):
		y[:,i] = input_data[n_inputs+gap+i:L-n_outputs+i]

	return X, y


In [ ]:
rate = web.DataReader(name='WGS10YR', data_source='fred', start='2000-01-01').diff().dropna()
rate.plot()
plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Переведем ряда массив Numpy
series = rate.squeeze().to_numpy()

In [ ]:
# задаём ширину окна и горизонт прогнозирования
n_lags, fh= 20, 10

X, y = make_datasets(series, n_inputs=n_lags, n_outputs=fh)

In [ ]:
X_tensor = torch.Tensor(X).to(device)
y_tensor = torch.Tensor(y).to(device)

dataset = TensorDataset(X_tensor, y_tensor)

# Для воспроизводимости задаём генератор
generator = torch.Generator().manual_seed(42)
# делим выборку на обучающую и тестовую с соотношении 80:20
train_dataset, test_dataset = random_split(dataset, [0.8, 0.2], generator=generator)

# Создаём DataLoader'ы
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()                                 # переводим молель в режим обучения
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()                     # Очищаем старые градиенты
        # Compute prediction error
        y_pred = model(X)                         # строми прооноз на батче
        loss = loss_fn(y_pred, y)                 # вычисляем ошибку проноза/значение функции потерь

        # Backpropagation
        loss.backward()                           # Вычисляем градиенты
        optimizer.step()                          # Обновляем веса по градиентам

        if batch % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: Avg loss = {test_loss:>8f} \n")

In [ ]:
# MLP с одним скрытым слоем
hidden_state = 30

model = nn.Sequential(
    nn.Linear(n_lags, hidden_state),
    nn.ReLU(),
    # nn.Sigmoid(),
    # nn.Tanh(),
    nn.Linear(hidden_state, fh)
).to(device)


In [ ]:
# Число параметров модели
for params in model.parameters():
  print(params.size())

In [ ]:
# Обучение
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 10

for epoch in range(epochs):
	print(f"Epoch {epoch+1}\n-------------------------------")
	train(train_dataloader, model, loss_fn, optimizer)
	# test(test_dataloader, model, loss_fn)

print("Done!")

In [ ]:
# метрики на тестово множества
test(test_loader, model, loss_fn)